In [ ]:
# Komórka 1: Konfiguracja Środowiska i Instalacja Zależności

import os

# Krok 1: Klonowanie repozytorium z modułami
print(" cloning repository...")
!git clone https://github.com/MattyMroz/ColabUniversalDownloader.git

# Przejście do katalogu repozytorium
os.chdir('ColabUniversalDownloader')
print(f" Zmieniono katalog roboczy na: {os.getcwd()}")

# Krok 2: Instalacja zależności systemowych (megatools)
print("\n⚙️ Instalacja 'megatools'...")
!sudo apt-get update -qq > /dev/null
!sudo apt-get install -y megatools -qq > /dev/null
print("✅ 'megatools' zainstalowane pomyślnie.")

# Krok 3: Instalacja zależności z pliku requirements.txt
print("\n🐍 Instalowanie bibliotek Pythona z requirements.txt...")
!pip install -q -r requirements.txt
print("✅ Biblioteki Pythona zainstalowane pomyślnie.")

print("\n\n🎉 Środowisko jest gotowe do testów!")

In [ ]:
# Komórka 2: Import Modułów i Inicjalizacja Narzędzi

import sys
from pprint import pprint

# Dodajemy ścieżkę do repozytorium, aby Python widział nasze moduły w katalogu 'utils'
sys.path.append('/content/ColabUniversalDownloader')

from utils.pixeldrain import PixelDrainDownloader
from utils.mega import MegaDownloader
from utils.google_drive import GoogleDriveManager

# --- Inicjalizacja klas ---
pixeldrain_handler = PixelDrainDownloader()
mega_handler = MegaDownloader()
gdrive_handler = GoogleDriveManager()

print("✅ Moduły zaimportowane, obiekty gotowe do pracy.")
print(f"Google Drive Manager gotowy: {gdrive_handler.is_ready()}")

# --- Prosta funkcja do wyświetlania postępu ---
def simple_progress_callback(log_message):
    """Wyświetla każdą nową linię logu."""
    print(log_message)

In [ ]:
# Komórka 3: Test Modułu `pixeldrain.py`

print("="*50)
print("🚀 TEST 1: POBIERANIE Z PIXELDRAIN 🚀")
print("="*50)

# UWAGA: Użyj działającego linku do testów
PIXEL_URL = "https://pixeldrain.com/u/e75isJ7y" # Przykładowy link, może wygasnąć
filepath = None

try:
    filepath = pixeldrain_handler.download(PIXEL_URL, simple_progress_callback)

    if filepath and os.path.exists(filepath):
        print(f"\n✅ TEST ZAKOŃCZONY SUKCESEM! Plik znajduje się w: {filepath}")
        print(f"   Rozmiar pliku: {os.path.getsize(filepath) / 1024:.2f} KB")
    else:
        print("\n❌ TEST NIEUDANY. Nie udało się pobrać pliku.")

finally:
    # Sprzątanie po teście
    if filepath and os.path.exists(filepath):
        os.remove(filepath)
    #     print(f"\n🧹 Usunięto plik testowy: {filepath}")

In [ ]:
# Komórka 4: Test Modułu `mega.py`

print("="*50)
print("🚀 TEST 2: OPERACJE NA MEGA.NZ 🚀")
print("="*50)

# --- Część A: Listowanie zawartości folderu ---
print("\n--- TEST 2A: Listowanie folderu ---")
# UWAGA: Użyj działającego linku do folderu
MEGA_FOLDER_URL = "https://mega.nz/folder/01ZzzS7Q#kmmLp-S_2k39y4p2z3HPUg"  # Przykładowy folder, może wygasnąć
folder_contents = None
try:
    folder_contents = mega_handler.list_folder_contents(MEGA_FOLDER_URL)
except Exception as e:
    print(f"❌ BŁĄD podczas listowania: {e}")

if folder_contents:
    print("\n✅ TEST ZAKOŃCZONY SUKCESEM! Zawartość folderu:")
    print(folder_contents)
else:
    print("\n⚠️ Brak danych do wyświetlenia (możliwy nieaktualny link lub błąd sieci)")

# --- Część B: Pobieranie jednego pliku ---
print("\n\n--- TEST 2B: Pobieranie pliku ---")
# UWAGA: Podmień na działający link do pliku Mega
MEGA_FILE_URL = "https://mega.nz/file/REPLACE#WITH_VALID_KEY"  # ← Podmień na poprawny URL
filepath = None

try:
    if "REPLACE" not in MEGA_FILE_URL:
        filepath = mega_handler.download_file(MEGA_FILE_URL, progress=simple_progress_callback)
        if filepath and os.path.exists(filepath):
            print(f"\n✅ POBRANO PLIK: {filepath}")
            print(f"   Rozmiar: {os.path.getsize(filepath) / (1024*1024):.2f} MB")
        else:
            print("\n❌ Nie udało się pobrać pliku (sprawdź URL).")
    else:
        print("UWAGA: Użyto placeholdera. Podmień MEGA_FILE_URL na prawidłowy link do pliku.")
finally:
    # (Opcjonalnie) Sprzątanie po teście
    pass

# --- Część C: Pobieranie folderu (opcjonalnie) ---
print("\n\n--- TEST 2C: Pobieranie folderu (opcjonalnie) ---")
CHOOSE_FILES = False  # Ustaw True, aby w terminalu wybrać pliki (może nie działać w Colab bez TTY)
DEST_DIR = "."      # Katalog docelowy

try:
    results = mega_handler.download_folder(MEGA_FOLDER_URL, dest_dir=DEST_DIR, choose_files=CHOOSE_FILES, progress=simple_progress_callback)
    if results:
        print("\n✅ POBRANE ELEMENTY:")
        for p in results:
            print(f" - {p}")
    else:
        print("\nℹ️ Brak zidentyfikowanych plików w logu lub folder pusty (jeśli link wygasł – podmień na aktualny).")
except Exception as e:
    print(f"❌ BŁĄD podczas pobierania folderu: {e}")

In [ ]:
# Komórka 5: Test Modułu `google_drive.py`

print("="*50)
print("🚀 TEST 3: OPERACJE NA GOOGLE DRIVE 🚀")
print("="*50)
print("UWAGA: Ta komórka wywoła okno autoryzacji Google. Zezwól na dostęp.")

if not gdrive_handler.is_ready():
    print("\n❌ TEST NIEUDANY. Manager Dysku Google nie został poprawnie zainicjalizowany.")
else:
    DUMMY_FILENAME = "test_upload.txt"
    DUMMY_CONTENT = "To jest plik testowy z Colab Universal Downloader."
    file_id_to_delete = None
    
    try:
        # Tworzenie pliku testowego
        with open(DUMMY_FILENAME, "w") as f:
            f.write(DUMMY_CONTENT)
        print(f"Stworzono plik testowy: {DUMMY_FILENAME}")

        # Krok 1: Pobierz ID "Mojego Dysku"
        print("\n--> Testowanie get_drive_id()...")
        parent_id = gdrive_handler.get_drive_id(drive_name="Mój Dysk", is_shared=False)
        assert parent_id == 'root'
        print("✅ ID 'Mojego Dysku' pobrane poprawnie ('root').")

        # Krok 2: Wyślij plik i udostępnij
        print("\n--> Testowanie upload_and_share()...")
        upload_info = gdrive_handler.upload_and_share(DUMMY_FILENAME, parent_id)
        
        if upload_info and upload_info.get('link'):
            print(f"✅ Plik wysłany i udostępniony pomyślnie!")
            print(f"   Link do pobrania: {upload_info['link']}")
            file_id_to_delete = upload_info['id']
        else:
            raise AssertionError("Nie udało się wysłać pliku lub uzyskać linku.")

        # Krok 3: Zaplanuj usunięcie pliku
        if file_id_to_delete:
            print("\n--> Testowanie delete_file_after_delay()...")
            gdrive_handler.delete_file_after_delay(file_id_to_delete, delay_seconds=15)
            print("✅ Zadanie usunięcia pliku zaplanowane na za 15 sekund.")
            print("   Możesz sprawdzić na Dysku Google, czy plik zniknie.")

    except Exception as e:
        print(f"\n❌ WYSTĄPIŁ BŁĄD PODCZAS TESTU: {e}")
    finally:
        # Sprzątanie lokalnego pliku
        if os.path.exists(DUMMY_FILENAME):
            os.remove(DUMMY_FILENAME)
            print(f"\n🧹 Usunięto lokalny plik testowy: {DUMMY_FILENAME}")